In [25]:
import numpy as np
import struct

In [26]:
def load_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, 1, 28, 28)
        return images / 255.0

def load_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
        return labels

In [27]:
# Load dataset from /content
X_train = load_images("/content/sample_data/train-images-idx3-ubyte")
y_train = load_labels("/content/sample_data/train-labels-idx1-ubyte")

X_test = load_images("/content/sample_data/t10k-images-idx3-ubyte")
y_test = load_labels("/content/sample_data/t10k-labels-idx1-ubyte")

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (60000, 1, 28, 28)
Testing shape: (10000, 1, 28, 28)


In [28]:
class ConvLayer:
    def __init__(self, num_filters, filter_size):
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.filters = np.random.randn(num_filters, 1, filter_size, filter_size) * 0.05

    def forward(self, input):
        self.input = input
        batch, _, h, w = input.shape
        self.output = np.zeros((batch, self.num_filters, h-2, w-2))

        for b in range(batch):
            for f in range(self.num_filters):
                for i in range(h-2):
                    for j in range(w-2):
                        region = input[b, :, i:i+3, j:j+3]
                        self.output[b, f, i, j] = np.sum(region * self.filters[f])
        return self.output

    def backward(self, d_out, lr):
        d_filters = np.zeros_like(self.filters)
        batch, _, h, w = self.input.shape

        for b in range(batch):
            for f in range(self.num_filters):
                for i in range(h-2):
                    for j in range(w-2):
                        region = self.input[b, :, i:i+3, j:j+3]
                        d_filters[f] += d_out[b, f, i, j] * region

        self.filters -= lr * d_filters

In [29]:
def relu(x):
    return np.maximum(0, x)

def relu_backward(d_out, x):
    d = d_out.copy()
    d[x <= 0] = 0
    return d

In [30]:
class MaxPool:
    def forward(self, input):
        self.input = input
        batch, filters, h, w = input.shape
        self.output = np.zeros((batch, filters, h//2, w//2))

        for b in range(batch):
            for f in range(filters):
                for i in range(0, h, 2):
                    for j in range(0, w, 2):
                        region = input[b, f, i:i+2, j:j+2]
                        self.output[b, f, i//2, j//2] = np.max(region)
        return self.output

    def backward(self, d_out):
        batch, filters, h, w = self.input.shape
        d_input = np.zeros_like(self.input)

        for b in range(batch):
            for f in range(filters):
                for i in range(0, h, 2):
                    for j in range(0, w, 2):
                        region = self.input[b, f, i:i+2, j:j+2]
                        max_val = np.max(region)

                        for x in range(2):
                            for y in range(2):
                                if region[x, y] == max_val:
                                    d_input[b, f, i+x, j+y] = d_out[b, f, i//2, j//2]

        return d_input

In [31]:
class FullyConnected:
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * 0.05
        self.bias = np.zeros(output_size)

    def forward(self, input):
        self.input = input
        return np.dot(input, self.weights) + self.bias

    def backward(self, d_out, lr):
        d_weights = np.dot(self.input.T, d_out)
        d_bias = np.sum(d_out, axis=0)
        d_input = np.dot(d_out, self.weights.T)

        self.weights -= lr * d_weights
        self.bias -= lr * d_bias

        return d_input

In [32]:
conv = ConvLayer(8, 3)
pool = MaxPool()
fc = FullyConnected(8 * 13 * 13, 10)

learning_rate = 0.003
epochs = 5
train_samples = 8000

In [33]:
# =============================
# SOFTMAX + LOSS FUNCTIONS
# =============================

def softmax(x):
    exp = np.exp(x - np.max(x))
    return exp / np.sum(exp)

def cross_entropy(pred, label):
    return -np.log(pred[label] + 1e-9)

In [34]:

# TRAINING


for epoch in range(epochs):

    correct = 0
    total_loss = 0

    for i in range(train_samples):

        image = X_train[i:i+1]
        label = y_train[i]

        # -------- Forward --------
        out_conv = conv.forward(image)
        out_relu = relu(out_conv)
        out_pool = pool.forward(out_relu)

        out_flat = out_pool.reshape(1, -1)
        out_fc = fc.forward(out_flat)

        probs = softmax(out_fc[0])

        loss = cross_entropy(probs, label)
        total_loss += loss

        if np.argmax(probs) == label:
            correct += 1

        # -------- Backward --------
        d_out = probs.copy()
        d_out[label] -= 1
        d_out = d_out.reshape(1, -1)

        d_fc = fc.backward(d_out, learning_rate)
        d_pool = d_fc.reshape(out_pool.shape)

        d_relu = pool.backward(d_pool)
        d_conv = relu_backward(d_relu, out_conv)

        conv.backward(d_conv, learning_rate)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print("Loss:", total_loss/train_samples)
    print("Training Accuracy:", correct/train_samples)


Epoch 1/5
Loss: 0.8857786484332862
Training Accuracy: 0.684

Epoch 2/5
Loss: 0.590078420833138
Training Accuracy: 0.795125

Epoch 3/5
Loss: 0.5062895921569562
Training Accuracy: 0.82475

Epoch 4/5
Loss: 0.4624143123627693
Training Accuracy: 0.83975

Epoch 5/5
Loss: 0.43340711986151004
Training Accuracy: 0.849875


In [35]:

# TESTING


correct = 0
test_samples = 2000   # testing on 2000 images for speed

for i in range(test_samples):

    image = X_test[i:i+1]
    label = y_test[i]

    out = conv.forward(image)
    out = relu(out)
    out = pool.forward(out)
    out = out.reshape(1, -1)
    out = fc.forward(out)

    probs = softmax(out[0])

    if np.argmax(probs) == label:
        correct += 1

print("\nFinal Test Accuracy:", correct/test_samples)


Final Test Accuracy: 0.8475
